In [2]:
import pandas as pd
import numpy as np
import pickle

oobasic = pd.read_excel('DATAFILES/oonames.xlsx',sheet_name='eco_oonames')
pickle.dump(oobasic, open('HOME_PICKLE_FILES/oonames.pkl','wb'))
instlist = pd.read_excel('DATAFILES/data_eco_bus.xlsx',sheet_name='listinstitutions')
pickle.dump(instlist, open('HOME_PICKLE_FILES/listinstitutions.pkl','wb'))

In [4]:
def matrixA(datalist, idatalist, journals, institutions):#### to transfer reputation from institutions to journals
    inst = institutions[['institution','acr']].copy()
    for k in range(len(datalist)):
        data = datalist[k]
        datacero = data.drop(['journal'], axis=1)
        datacero['ninst'] = 1/datacero['ninst']
        sj =  datacero.groupby('institution')['ninst'].sum()
        nsj = pd.DataFrame(sj)
        nsj = nsj.reset_index()
        nsj = nsj.rename(columns={'ninst':k})
        nsj = institutions.merge(nsj, on='institution',how='left').fillna(0)
        nsj = nsj[['acr',k]]
        nsj = nsj.sort_values(by=['acr'],ascending=True)
        inst = inst.merge(nsj,on='acr',how='left').fillna(0)
    inst = inst.drop(['institution', 'acr'], axis=1)
    inst['suma']=np.maximum(inst.sum(axis=1),1)
    ninst = inst.div(inst.suma,axis=0)#####for use of s_i
    ninst = ninst.drop(['suma'], axis=1)
    ninst = ninst.T
    A = ninst.to_numpy()
    return A, ninst

In [5]:
def matrixB(datalist, idatalist, journals, institutions):#### to transfer reputation from journals to institutions
    jour= journals[['journal','acr']].copy()
    for k in range(len(idatalist)):
        data = idatalist[k]
        datacero = data[['UT', 'journal']]
        datacero = datacero.drop_duplicates(subset=['UT', 'journal'], keep='first')
        sj =  datacero.groupby('journal')['UT'].count()
        nsj = pd.DataFrame(sj)
        nsj = nsj.reset_index()
        nsj =nsj.rename(columns={'UT':k})
        nsj = journals.merge(nsj, on='journal',how='left').fillna(0)
        nsj = nsj[['acr',k]]
        nsj = nsj.sort_values(by=['acr'],ascending=True)
        jour = jour.merge(nsj,on='acr',how='left').fillna(0)
    jour = jour.drop(['journal', 'acr'], axis=1)
    jour['suma']=np.maximum(jour.sum(axis=1),1)
    njour = jour.div(jour.suma,axis=0)##### for use of s_i
    njour = njour.drop(['suma'], axis=1)
    njour = njour.T
    B = njour.to_numpy()
    return B, njour

In [6]:
def updatescores(journals, institutions,  A, B):### computing the first eigenvector of BA
    v = institutions['iscore'].to_numpy()
    oldnorma = 10
    norma = 1
    while np.abs(oldnorma-norma) > 0.000000001:
        oldnorma = norma
        v = np.matmul(np.matmul(B,A),v)
        norma = np.linalg.norm(v)
        v = v/norma
        print(np.round(norma,10)) ### To witness how the convergence to the first eigenvalue takes place, we follow the value of the norm of the vector.
    w = np.matmul(A,v)### the first eigenvector of AB
    return v, w

In [16]:
jourdatalist= []
instdatalist = []
fields = {'eco_bus'}
#fields = {'man'}
for field in fields:
    print(field)
    #get the files needed for the algorithm from two folders: DATAFILES AND HOME_PICKLE_FILES
    #HOME_PICKLE_FILES STORES THE INFORMATION GATHERED FROM THE CITATION REPORTS OF JOURNALS AND INSTITUTIONS
    #OTHER JUNYPER NOTEBOOKS PREPARE THE INFORMATION AND STORE IT AS PICKLE FILES (EASY TO MANAGE WITHIN PYTHON)
    file = 'DATAFILES/data_'+ field+ '.xlsx'
    xl = pd.ExcelFile(file)
    d = {} # your dict.
    for sheet in xl.sheet_names:
        d[f'{sheet}']= pd.read_excel(xl,sheet_name=sheet)
        pickle.dump(d[f'{sheet}'], open('HOME_PICKLE_FILES/' + sheet + '.pkl','wb'))
    idatalist = pd.read_pickle('HOME_PICKLE_FILES/idatalist.pkl')
    datalist = pd.read_pickle('HOME_PICKLE_FILES/jdatalist.pkl')
    institutions = pd.read_pickle("HOME_PICKLE_FILES/institutions.pkl")
    journals = pd.read_pickle("HOME_PICKLE_FILES/journals.pkl")
    
    A, ninst = matrixA(datalist, idatalist, journals, institutions)### CITATION MATRIX FROM INSTITUTIONS TO JOURNALS
    B, njour = matrixB(datalist, idatalist, journals, institutions)### CITATION MATRIX FROM JOURNALS TO INSTITUTIONS
    v, w = updatescores(journals, institutions, A, B)### SIZE-DEPENDENT INFLUENCE EIGENVECTORS FOR INSTITUTIONS AND JOURNALS
    
    df = pd.DataFrame(v, columns=['iscore'])
    inst = institutions[['institution','pub','acr']].copy()
    inst = inst.join(df)
    inst['iscore'] = inst['iscore']/inst['pub']#### SIZE-INDEPENDENT INFLUENCE PER PAPER
    a = 7/( inst['iscore'].max()-inst['iscore'].min())
    inst['iscore'] =  1+a*(inst['iscore']-inst['iscore'].min())
 
    df = pd.DataFrame(w, columns=['journal_score'])
    jour = journals[['journal','acr']].copy()
    jour = journals[['journal','acr','pub']].copy()
    jour = jour.join(df)
    jour['journal_score'] = jour['journal_score']/jour['pub']#### SIZE-INDEPENDENT INFLUENCE PER PAPER
    a = 9.99/( jour['journal_score'].max()-jour['journal_score'].min())
    jour['journal_score'] =  0.01+a*(jour['journal_score']-jour['journal_score'].min())

    inst = inst.sort_values(by=['iscore'],ascending=False)
    jour = jour.sort_values(by=['journal_score'],ascending=False)
    jourdatalist.append(jour)
    instdatalist.append(inst)   

eco_bus
37.8265029583
1.1220584876
1.0190040713
0.9875840552
0.9774777841
0.9741764514
0.973092767
0.972736476
0.9726192745
0.9725807146
0.9725680274
0.972563853
0.9725624794
0.9725620275
0.9725618788
0.9725618299
0.9725618138
0.9725618085
0.9725618067
0.9725618061


In [20]:
filename = 'RESULTS/jirc_journal_institution_results.xlsx'
with pd.ExcelWriter(filename) as writer:
    for k, field in zip(range(len(jourdatalist)),fields):
        jourdatalist[k].to_excel(writer,sheet_name= 'journals')
        instdatalist[k].to_excel(writer,sheet_name= 'institutions')